# Local Food — Agente de planeamento de rotas

**Grupo 10** 
Thifany Antoni 16077
Pedro Perez 18623

## 1. Introdução

### Contexto do Problema
O aumento da procura na União Europeia por produtos alimentares locais e de proximidade tem esbarrado em grandes desafios para os pequenos produtores, nomeadamente os elevados custos logísticos e a volatilidade dos preços. Embora as cadeias de abastecimento curtas sejam fundamentais para reduzir as emissões de transporte e o desperdício alimentar, a subida acentuada dos preços das frutas e legumes afeta negativamente tanto quem produz como quem consome.

Para colmatar esta lacuna, este projeto foca-se no desenvolvimento de uma plataforma digital inovadora que interliga três agentes principais: Produtores (pequenas explorações locais), Clientes (que realizam encomendas e escolhem o local de entrega) e Centros de Recolha / Armazéns (responsáveis por agregar os produtos e gerir uma frota de veículos de distribuição).

### Propósito do Projeto
O objetivo central do trabalho é otimizar a distribuição local na região do distrito de Braga (incluindo a Póvoa de Varzim). O grande desafio reside no desenvolvimento de um agente de rotas capaz de selecionar os caminhos de recolha e entrega mais eficientes. Para isso, o sistema terá de processar e satisfazer simultaneamente várias restrições complexas:
*   Janelas temporais de disponibilidade: O período em que o produto está pronto no produtor.
*   Janelas temporais de entrega: Os prazos limite definidos pelos clientes para a receção da encomenda.
*   Capacidade dos veículos: A limitação física de carga da frota.

Para resolver este problema de otimização e planeamento, serão aplicadas e comparadas diferentes abordagens de Inteligência Artificial, nomeadamente algoritmos de Procura Informada (ex: $A^*$, Greedy Best-First), algoritmos de Procura Local (ex: Hill Climbing, Simulated Annealing) e técnicas de Problemas de Satisfação de Restrições (CSPs).

Nesta fase inicial de Análise Exploratória de Dados , o foco está na validação da integridade dos dados sintéticos fornecidos (matrizes de distância, localizações e encomendas) para garantir a consistência das restrições antes da fase de modelação.

## Entrega preliminar
As células seguintes verificam os dados e demonstram uma rota de um único item, incluindo exemplos intencionalmente inválidos. **Esses exemplos não são as soluções completas**; os planos completos são calculados na segunda parte, abaixo.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
assert (ROOT / 'src' / 'model.py').exists(), 'Abrir a pasta do projeto no VS Code.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.model import Problem, Request, Vehicle, State, Stop, Route, example_route
from src.analysis import audit, individual_feasibility
problem = Problem(ROOT / 'data_tiny')


In [ ]:
display(audit(problem))
print('Encomendas:',len(problem.tables['orders']),'| Itens:',len(problem.requests))

In [ ]:
examples=[(i,v,example_route(problem,i,v)) for i in problem.requests for v in problem.vehicles]
item_id,vehicle_id,route=next((i,v,r) for i,v,r in examples if problem.validate([r],require_all=False).feasible)
result=problem.validate([route],require_all=False)
print('Item:',item_id,'Veículo:',vehicle_id,'Viável:',result.feasible)
print('Custo em km:',round(problem.cost([route]),2))
display(result.timeline)
assert result.feasible
assert abs(result.cost_km-problem.cost([route]))<1e-8

In [ ]:
req=problem.requests[item_id]
v=problem.vehicles[vehicle_id]
bad=Route(vehicle_id,route.start,[Stop(req.pickup,'pickup',(item_id,)),Stop(req.delivery,'deliver',(item_id,)),Stop(v.depot,'visit')])
bad_result=problem.validate([bad],require_all=False)
print('Viável:',bad_result.feasible)
print('\n'.join(bad_result.errors))
assert not bad_result.feasible

In [ ]:
complete=problem.validate([route])
print('Satisfaz todos os pedidos?',complete.feasible)
print('Primeiros avisos:',complete.errors[:3])
assert not complete.feasible

In [ ]:
import subprocess
completed=subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=ROOT,capture_output=True,text=True)
print(completed.stdout+completed.stderr)
assert completed.returncode==0

# Planeamento completo

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.model import Problem
from src.planning import Planner
from src.informed import search
from src.local_search import hill_climbing
from src.csp import solve_csp
from time import perf_counter
problem = Problem(ROOT / 'data_tiny')
t0 = perf_counter()
planner = Planner(problem)
preparation_seconds = perf_counter() - t0
print(f'Preparação comum: {preparation_seconds:.4f} s')
print('Lotes por onda:', [len(w) for w in planner.waves])
print('Valores por domínio:', [len(o) for o in planner.options])

## Formulação e limites da pesquisa

Usamos o custo total em **quilómetros**. Cada linha de `order_items` é indivisível, mas linhas da mesma encomenda podem viajar separadamente. Não inventamos exigências de refrigeração, turnos ou limites de cais que não constam dos dados. A matriz fornecida determina distância e tempo; não usamos serviços externos.

Para tornar a comparação executável, restringimos o espaço de pesquisa a **cinco ondas**: recolhas de 30/10, entregas de 30/10, recolhas de 31/10, entregas de 31/10 recolhidas na véspera e restantes entregas de 31/10. Cada veículo executa as ondas por esta ordem, com relógio próprio; não precisa de esperar que o outro termine a onda. Itens do mesmo produtor/dia são agrupados; entregas são agrupadas por destino/dia e por recolha na véspera ou no próprio dia. A janela do lote é a interseção das janelas dos seus itens. Um lote não é dividido entre veículos. Se a interseção for vazia, o programa informa que esta formulação exige divisão do lote.

Uma decisão escolhe a distribuição dos lotes da onda pelos dois veículos e a ordem das visitas. Cada veículo faz no máximo uma viagem por onda, partindo e regressando ao armazém. Nas recolhas descarrega tudo ao regressar; nas entregas carrega antes de partir. Um veículo pode ficar parado numa onda. Estas escolhas são **hipóteses de implementação**, não exigências adicionais do enunciado.

O estado contém a onda seguinte, as duas rotas parciais, o instante de disponibilidade de cada veículo e os instantes de chegada e descarga de cada item no armazém. A disponibilidade para carregamento começa no fim da descarga. A permanência máxima de 24 h conta desde a chegada ao armazém até ao início do carregamento. Cada visita a produtor/cliente demora 10 min e cada carga/descarga demora 15 min. As janelas referem-se ao início do serviço. Esperas são permitidas; no carregamento esperamos pela partida útil para a primeira entrega.

Os sucessores enumeram todas as permutações e todos os cortes em duas sequências, rejeitando excessos de capacidade e violações temporais. O objetivo exige todos os itens entregues, regresso ao armazém e validação integral por `Problem.validate(require_all=True)`, incluindo inventário do armazém e precedências. A capacidade do armazém é verificada na solução completa, não usada como poda intermédia.

**Limite:** as conclusões de otimalidade de A* referem-se apenas a este espaço de ondas, lotes e horários determinísticos. Não se demonstra o ótimo do problema geral; falhar aqui também não prova que o problema geral é inviável. `data_small` permanece na EDA; esta comparação usa os dois veículos de `data_tiny`, como solicitado.

## Procura informada: Greedy e A*

`g(n)` é a distância das ondas já decididas. Para cada onda futura calculamos dois limites inferiores:

- **h1:** maior distância de ida e volta ao armazém para um dos seus lotes, limitada superiormente pelo mínimo geométrico da onda. Somamos estes valores nas ondas restantes.
- **h2:** custo mínimo entre todas as distribuições e ordens de uma onda, respeitando capacidade, mas ignorando horários, stock e ligação temporal às outras ondas. Somamos estes mínimos nas ondas restantes.

A limitação em h1 evita depender de desigualdades triangulares exatas numa matriz arredondada. Por construção, `0 ≤ h1 ≤ h2 ≤ custo restante` no espaço definido. Ambas são consistentes: a onda escolhida custa pelo menos o respetivo limite. h2 é mais informativa, mas exige enumerar as opções na preparação comum. Essa preparação é medida separadamente e usada em todas as execuções.

**Greedy Best-First** usa apenas `f=h`; **A*** usa `f=g+h`. A fila é um heap. Nos empates preferimos maior profundidade e, depois, a ordem de geração. Como as heurísticas dependem da onda e não da distribuição já escolhida, Greedy pode produzir exatamente a mesma solução com h1 e h2; esta é uma limitação destas heurísticas para Greedy. A* usa também o custo acumulado.

Contamos como expandido um nó não terminal retirado da fila cujos sucessores são examinados. Nós gerados incluem a raiz e os sucessores viáveis inseridos na fila. Não há fusão de estados nem conjunto fechado: pesquisamos uma árvore finita de profundidade cinco. O limite de 20 000 expansões é indicado como interrupção, nunca como solução ou prova de inviabilidade.

In [ ]:
informed_results = [search(planner, algorithm, heuristic)
                    for algorithm in ('greedy', 'astar') for heuristic in ('h1', 'h2')]
for result in informed_results:
    assert result.routes is not None, (result.algorithm, result.status)
    check = problem.validate(result.routes)
    assert check.feasible, check.errors
rows = [dict(metodo=r.algorithm, km=r.cost_km, nos_expandidos=r.expanded,
             nos_gerados=r.generated, segundos=r.seconds, estado=r.status)
        for r in informed_results]
display(pd.DataFrame(rows).round(4))

## Procura local: Hill Climbing

A solução inicial é a de Greedy h1. Um vizinho substitui a distribuição e a ordem de **uma** onda, mantendo as restantes. Recalculamos os horários posteriores e rejeitamos soluções que violem qualquer regra. Escolhemos o melhor vizinho de custo estritamente menor; terminamos quando não há melhoria ou após 50 melhorias. Não aceitamos movimentos laterais. A curva inclui a solução inicial na iteração zero.

O resultado é um ótimo local nesta vizinhança; Hill Climbing não garante o ótimo global. Neste ensaio pode coincidir com A*, o que não generaliza para outros dados. O tempo da pesquisa local exclui a obtenção da solução inicial; mostramos também o tempo total Greedy + Hill Climbing.

In [ ]:
greedy = informed_results[0]
local_result, history = hill_climbing(planner, greedy)
assert problem.validate(local_result.routes).feasible
print('Histórico de custos (km):', [round(x, 2) for x in history])
print(f'Tempo local: {local_result.seconds:.4f} s; com Greedy: {greedy.seconds + local_result.seconds:.4f} s')
plt.figure(figsize=(8, 4))
plt.plot(range(len(history)), history, marker='o')
plt.xlabel('Iteração (0 = Greedy)'); plt.ylabel('Distância total (km)')
plt.title('Hill Climbing — evolução do custo'); plt.xticks(range(len(history)))
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## CSP: variáveis, domínios e restrições

Cada variável `X_d` representa a decisão de uma onda. O seu domínio contém pares de sequências ordenadas de lotes, uma por veículo. Os domínios têm, neste conjunto, 18, 120, 24, 24 e 120 valores. A atribuição das cinco variáveis determina rotas e horários.

**Restrições duras:** cada lote é atribuído uma vez; capacidades dos veículos; janelas de recolha e entrega; encomenda anterior à recolha; fluxo produtor → armazém → cliente; carga após descarga; permanência de até 24 h; capacidade e inventário não negativo do armazém; horizonte e regresso ao depósito. A validação final confirma todas as restrições do modelo.

Usamos backtracking com ordem temporal fixa das variáveis, pois o relógio e stock de uma onda dependem das anteriores. O forward checking exige que a próxima onda conserve pelo menos um valor compatível; não faz propagação completa. Não afirmamos usar MRV. As variáveis das ondas têm restrições entre si, não são cinco problemas independentes.

A **preferência suave** é percorrer menos quilómetros: os valores são tentados por distância crescente. Isso orienta a pesquisa, mas não permite violar restrições duras. O CSP devolve a primeira atribuição viável, não uma prova de custo mínimo. Um backtrack é contado quando um valor localmente viável é abandonado por falta de continuação (incluindo poda pelo forward checking). Zero backtracks significa que o caminho escolhido não exigiu recuo, não que faltou implementar backtracking.

In [ ]:
csp_result, csp_stats = solve_csp(planner)
assert csp_result.routes is not None, csp_result.status
validation = problem.validate(csp_result.routes)
assert validation.feasible, validation.errors
print('Custo:', round(csp_result.cost_km, 2), 'km')
print('Estatísticas:', csp_stats)
print(f'Tempo: {csp_result.seconds:.4f} s')
assignment = []
for d, option in enumerate(csp_result.choices):
    for vehicle_index, sequence in enumerate(option):
        assignment.append(dict(onda=d+1, veiculo=planner.vehicles[vehicle_index].id,
                              locais=[planner.waves[d][k].location for k in sequence],
                              itens=[i for k in sequence for i in planner.waves[d][k].items]))
display(pd.DataFrame(assignment))

## Resultados, rotas e discussão

A tabela compara distância, tempo da pesquisa e validação integral. Os tempos dependem do computador e não incluem a preparação comum. O tempo de Hill Climbing abaixo inclui o Greedy inicial. As contagens de expansões de A*/Greedy, melhorias de Hill Climbing e atribuições do CSP têm significados diferentes; não devem ser comparadas como se fossem a mesma medida.

Nos resultados de referência, Greedy percorre 756,61 km. Hill Climbing reduz para 580,90 km em cinco melhorias, uma redução de cerca de 23,22%. A* com ambas as heurísticas encontra 580,90 km; h2 expande muito menos nós do que h1 (6 contra 4 273 neste ensaio). O CSP também encontra 580,90 km e não necessita de recuos nesta ordenação. Os valores efetivos são recalculados abaixo.

A* h2 é a melhor escolha neste ensaio pelo custo e pelo baixo número de expansões. Contudo, a preparação enumera permutações e cresce rapidamente com o número de locais; estes resultados não demonstram escalabilidade. O CSP encontra uma solução rapidamente, mas a ordenação por custo não garante o ótimo. A distância é uma aproximação do esforço de transporte, não uma medida direta de emissões ou de custo monetário.

Como melhorias futuras: dividir lotes quando necessário; permitir várias viagens e intercalar recolhas/entregas livremente; testar outras heurísticas mais discriminantes para Greedy; usar outras vizinhanças e reinícios na pesquisa local; estudar o conjunto `small`; medir tempos em várias repetições. Nenhuma destas extensões é apresentada como já implementada.

In [ ]:
all_results = informed_results + [local_result, csp_result]
summary = pd.DataFrame([dict(metodo=r.algorithm, km=r.cost_km,
    segundos=r.seconds + (greedy.seconds if r is local_result else 0),
    viavel=problem.validate(r.routes).feasible) for r in all_results])
display(summary.round(4))
print(f'Redução Greedy → Hill Climbing: {100*(greedy.cost_km-local_result.cost_km)/greedy.cost_km:.2f}%')
best = min(all_results, key=lambda r:r.cost_km)
validation = problem.validate(best.routes)
print('Melhor plano:', best.algorithm, '| Todos os 40 itens:', validation.feasible)
for route in best.routes:
    print(f'Veículo {route.vehicle_id}: {problem.cost([route]):.2f} km')
    display(validation.timeline[validation.timeline.vehicle == route.vehicle_id])
output = ROOT / 'results'; output.mkdir(exist_ok=True)
summary.to_csv(output/'comparison.csv', index=False)
validation.timeline.to_csv(output/'best_routes.csv', index=False)
pd.DataFrame({'iteracao':range(len(history)), 'km':history}).to_csv(output/'hill_climbing.csv', index=False)
plt.figure(figsize=(9,4)); plt.bar(summary.metodo, summary.km)
plt.ylabel('Distância total (km)'); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## Conclusão

O projeto integra leitura e análise dos CSV, modelação, quatro abordagens de IA e validação independente das rotas. No conjunto tiny, as soluções apresentadas satisfazem todos os itens e as regras modeladas, com utilização dos dois veículos. A pesquisa informada mostra o efeito de uma heurística mais forte; a pesquisa local melhora uma solução inicial; o CSP evidencia a diferença entre satisfazer restrições e provar otimalidade.

Ferramentas: Python, pandas para tabelas, matplotlib para gráficos, Jupyter para documentação executável, unittest para verificação e Git/GitHub para colaboração. A preparação teve apoio de IA; os autores devem rever a formulação, compreender o código e respeitar as regras da unidade curricular sobre a declaração desse apoio. A validade dos planos é verificada automaticamente, mas a adequação das hipóteses deve ser discutida na defesa.